# LeetCode 2824 — Count Pairs Whose Sum is Less than Target bằng Python

Link đề: https://leetcode.com/problems/count-pairs-whose-sum-is-less-than-target/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def countPairs(self, nums: List[int], target: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số cặp chỉ số `(i, j)` với `0 <= i < j < n` sao cho `nums[i] + nums[j] < target`.
2. Biết **tự tính tay** bằng cách liệt kê cặp, rồi nhận ra rằng sau khi **sắp xếp** thì việc đếm có thể "nhảy cả cụm" thay vì đếm từng cặp.
3. Học các công cụ Python: vòng `for` lồng nhau, `range(i + 1, n)`, `sorted`, **hai con trỏ** (`left`, `right`), `itertools.combinations`, `bisect_left`.
4. Đi từ **Brute Force `O(n²)`** → **`combinations` một dòng** → **sắp xếp + hai con trỏ `O(n log n)`** → **sắp xếp + tìm kiếm nhị phân `O(n log n)`**.
5. Chứng minh vì sao hai con trỏ đúng ("nếu `nums[l] + nums[r] < target` thì mọi phần tử từ `l+1..r` cũng ghép được với `l`") và tránh các lỗi kinh điển: đếm cặp `(i, i)`, đếm cặp hai lần, quên rằng sắp xếp không làm đổi số cặp.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` (0-indexed) có độ dài `n` và một số nguyên `target`. Trả về **số cặp** `(i, j)` thoả mãn đồng thời:

- `0 <= i < j < n`
- `nums[i] + nums[j] < target`

| Ví dụ | `nums`, `target` | Output | Giải thích |
|---|---|---|---|
| 1 | `[-1, 1, 2, 3, 1]`, `2` | `3` | Các cặp thoả mãn: `(0, 1)` vì `-1 + 1 = 0 < 2`; `(0, 2)` vì `-1 + 2 = 1 < 2`; `(0, 4)` vì `-1 + 1 = 0 < 2`. Cặp `(0, 3)` có tổng `2` không `< 2`. Tổng cộng 3 cặp. |
| 2 | `[-6, 2, 5, -2, -7, -1, 3]`, `-2` | `10` | Có đúng 10 cặp có tổng nhỏ hơn `-2` (ta sẽ liệt kê ngay bên dưới để tự kiểm). |

Constraints:

- `1 <= nums.length == n <= 50`
- `-50 <= nums[i], target <= 50`

In [1]:
# Liệt kê tất cả cặp thoả mãn của hai ví dụ để tự kiểm tra bằng mắt
def liet_ke(nums, target):
    res = []
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] < target:
                res.append((i, j, nums[i] + nums[j]))
    return res

print(liet_ke([-1, 1, 2, 3, 1], 2))
print(len(liet_ke([-6, 2, 5, -2, -7, -1, 3], -2)))

[(0, 1, 0), (0, 2, 1), (0, 4, 0)]
10


## 3. Kiến thức nền

### 3.1. Cặp `(i, j)` với `i < j` — vòng `for` lồng nhau

Để xét **mỗi cặp đúng một lần**, ta cho `i` chạy `0..n-1` và `j` chạy `i+1..n-1`. Điều kiện `i < j` bảo đảm không xét `(i, i)` và không xét cùng cặp hai lần theo hai thứ tự `(i, j)`, `(j, i)`. Số cặp là `n(n−1)/2`.

In [2]:
n = 4
cap = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(cap)            # [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
print(len(cap), n * (n - 1) // 2)   # 6 6

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
6 6


### 3.2. Điều kiện trong `sum(...)`

Trong Python, biểu thức so sánh như `a + b < t` cho `True` hoặc `False`; `True` được tính là `1`, `False` là `0`. Vì vậy `sum(điều_kiện for ...)` **đếm** số lần điều kiện đúng. Đây là mẹo đếm rất hay dùng.

In [3]:
print(True + True + False)                       # 2
print(sum(x > 2 for x in [1, 2, 3, 4]))          # 2  (đếm số phần tử > 2)

2
2


### 3.3. `itertools.combinations(iterable, 2)`

`combinations(nums, 2)` sinh **mọi cặp** hai phần tử ở hai vị trí khác nhau, mỗi cặp **một lần**, theo thứ tự vị trí (tức đúng các cặp `i < j`).

In [4]:
from itertools import combinations

print(list(combinations([10, 20, 30], 2)))      # [(10, 20), (10, 30), (20, 30)]
print(len(list(combinations(range(5), 2))))     # 10

[(10, 20), (10, 30), (20, 30)]
10


### 3.4. Sắp xếp: `sorted` và `list.sort`

`sorted(nums)` trả list mới **tăng dần**; `nums.sort()` sắp xếp **tại chỗ**. Quan trọng: bài này đếm theo **giá trị cặp**, không theo chỉ số ban đầu, nên **sắp xếp không làm đổi đáp án** (mỗi cặp phần tử vẫn còn nguyên, chỉ đổi tên chỉ số).

In [5]:
nums = [-1, 1, 2, 3, 1]
print(sorted(nums))       # [-1, 1, 1, 2, 3]
nums.sort()
print(nums)               # [-1, 1, 1, 2, 3]

[-1, 1, 1, 2, 3]
[-1, 1, 1, 2, 3]


### 3.5. Hai con trỏ (two pointers)

Với mảng **đã sắp xếp**, đặt `left = 0` (phần tử nhỏ nhất) và `right = n − 1` (phần tử lớn nhất). Xét tổng `nums[left] + nums[right]`:

- Nếu tổng **quá lớn** (`>= target`), `nums[right]` quá lớn với mọi phần tử từ `left` trở đi ⇒ bỏ `right` (`right -= 1`).
- Nếu tổng **đủ nhỏ**, thì `nums[left]` ghép được với **mọi** phần tử `left+1..right` ⇒ đếm cả cụm một lần rồi `left += 1`.

In [6]:
nums = [-1, 1, 1, 2, 3]
left, right = 0, len(nums) - 1
print(nums[left], nums[right], nums[left] + nums[right])   # -1 3 2

-1 3 2


### 3.6. `bisect_left(a, x, lo)` — tìm kiếm nhị phân

Với `a` **đã sắp xếp**, `bisect_left(a, x)` trả về vị trí **đầu tiên** mà chèn `x` vào vẫn giữ thứ tự, tức là **số phần tử `< x`** (nếu `lo = 0`). Tham số `lo` bắt đầu tìm từ vị trí `lo`. Độ phức tạp `O(log n)`.

In [7]:
from bisect import bisect_left

a = [-1, 1, 1, 2, 3]
print(bisect_left(a, 2))        # 3  (có 3 phần tử < 2: -1, 1, 1)
print(bisect_left(a, 1))        # 1  (chỉ -1 < 1)
print(bisect_left(a, 2, 2))     # 3  (tìm từ vị trí 2 trở đi)

3
1
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1: `nums = [-1, 1, 2, 3, 1]`, `target = 2`.

Cách làm tự nhiên: viết ra **mọi cặp** và kiểm tổng.

| cặp `(i, j)` | `nums[i]` | `nums[j]` | tổng | `< 2`? |
|---|---|---|---|---|
| (0, 1) | −1 | 1 | 0 | có |
| (0, 2) | −1 | 2 | 1 | có |
| (0, 3) | −1 | 3 | 2 | không |
| (0, 4) | −1 | 1 | 0 | có |
| (1, 2) | 1 | 2 | 3 | không |
| (1, 3) | 1 | 3 | 4 | không |
| (1, 4) | 1 | 1 | 2 | không |
| (2, 3) | 2 | 3 | 5 | không |
| (2, 4) | 2 | 1 | 3 | không |
| (3, 4) | 3 | 1 | 4 | không |

Có **3** cặp thoả. Với `n = 5` ta kiểm `10` cặp. Với `n = 50` là `1225` cặp, vẫn nhỏ.

**Nhận xét hướng tối ưu:** nếu **sắp xếp** mảng thành `[-1, 1, 1, 2, 3]`, thì với phần tử `-1`: các phần tử `1, 1, 2` ghép được (tổng lần lượt `0, 0, 1`, đều `< 2`), còn `3` thì không (`-1 + 3 = 2` không `< 2`). Các phần tử ghép được với `-1` luôn là một **đoạn liên tiếp ngay sau nó** (`1, 1, 2`). Khi đó ta đếm cả đoạn mà không cần xét từng cặp.

In [8]:
a = sorted([-1, 1, 2, 3, 1])
target = 2
print(a)
for j in range(1, len(a)):
    print(f"-1 + a[{j}]={a[j]} -> tổng {a[0] + a[j]} {'<' if a[0] + a[j] < target else '>='} {target}")

[-1, 1, 1, 2, 3]
-1 + a[1]=1 -> tổng 0 < 2
-1 + a[2]=1 -> tổng 0 < 2
-1 + a[3]=2 -> tổng 1 < 2
-1 + a[4]=3 -> tổng 2 >= 2


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp lồng nhau

**Ý tưởng:** làm đúng như đề. Với mỗi `i`, duyệt `j` từ `i + 1` tới `n − 1`; nếu `nums[i] + nums[j] < target` thì tăng `count`.

In [9]:
from typing import List

def count_pairs_bruteforce(nums: List[int], target: int) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] + nums[j] < target:
                count += 1
    return count

In [10]:
print(count_pairs_bruteforce([-1, 1, 2, 3, 1], 2))              # 3
print(count_pairs_bruteforce([-6, 2, 5, -2, -7, -1, 3], -2))    # 10
print(count_pairs_bruteforce([5], 10))                          # 0  (một phần tử: không có cặp)

3
10
0


**Độ phức tạp:** số cặp `n(n−1)/2` ⇒ Time `O(n²)`. Space `O(1)`.

**Nút thắt:** với `n <= 50` thì nhanh thừa. Nhưng nếu `n = 10^5`, `O(n²)` là quá chậm. Điều ta bỏ phí: sau khi đã biết `nums[i] + nums[j] < target` với `j` này, mọi `j' < j` (trong mảng đã sắp xếp) cũng thoả; ta lại kiểm từng cái.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — sắp xếp không đổi đáp án.** Đề đếm cặp vị trí khác nhau theo tổng giá trị. Sắp xếp chỉ hoán vị các phần tử; tập các cặp (theo phần tử) không đổi. Nên ta được tự do sắp xếp.

**Đột phá 2 — trong mảng tăng dần, điều kiện "tổng `< target`" có tính đơn điệu.** Nếu `a[l] + a[r] < target` thì với mọi `m` thoả `l < m <= r`: `a[l] + a[m] <= a[l] + a[r] < target`. Tức `a[l]` ghép được với **cả đoạn** `l+1..r`, gồm `r − l` phần tử.

**Đột phá 3 — hai con trỏ.** Đặt `l = 0, r = n−1`.

- Nếu `a[l] + a[r] < target`: cộng `r − l` cặp (tất cả cặp `(l, m)` với `l < m <= r`), rồi `l += 1`.
- Ngược lại: `a[r]` quá lớn với mọi `l' >= l` (vì `a[l'] >= a[l]`), nên **mọi cặp chứa `r` với phần tử từ `l` tới `r−1` đều hỏng**; bỏ `r`: `r -= 1`.

Mỗi bước dịch ít nhất một con trỏ ⇒ tối đa `n` bước.

In [11]:
a = sorted([-1, 1, 2, 3, 1])      # [-1, 1, 1, 2, 3]
target = 2
l, r = 0, len(a) - 1
count = 0
while l < r:
    s = a[l] + a[r]
    if s < target:
        count += r - l
        print(f"l={l}, r={r}: {a[l]}+{a[r]}={s} < {target}: cộng {r - l} cặp")
        l += 1
    else:
        print(f"l={l}, r={r}: {a[l]}+{a[r]}={s} >= {target}: bỏ r")
        r -= 1
print("count =", count)    # 3

l=0, r=4: -1+3=2 >= 2: bỏ r
l=0, r=3: -1+2=1 < 2: cộng 3 cặp
l=1, r=3: 1+2=3 >= 2: bỏ r
l=1, r=2: 1+1=2 >= 2: bỏ r
count = 3


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — `combinations` và `sum` (Pythonic brute force)

**Từng đoạn:**

- `combinations(nums, 2)` sinh mọi cặp `(a, b)` ở hai vị trí khác nhau.
- `a + b < target` cho `True` / `False`; `sum(...)` đếm số `True`.

Vẫn là `O(n²)`, nhưng ngắn và dễ đọc; chỉ nên dùng khi `n` nhỏ.

In [12]:
from itertools import combinations

def count_pairs_combinations(nums: List[int], target: int) -> int:
    return sum(a + b < target for a, b in combinations(nums, 2))

In [13]:
print(count_pairs_combinations([-1, 1, 2, 3, 1], 2))              # 3
print(count_pairs_combinations([-6, 2, 5, -2, -7, -1, 3], -2))    # 10

3
10


**Code nộp LeetCode (Cách 2):**

```python
from typing import List
from itertools import combinations

class Solution:
    def countPairs(self, nums: List[int], target: int) -> int:
        return sum(a + b < target for a, b in combinations(nums, 2))
```

**Độ phức tạp:** Time `O(n²)`, Space `O(1)` (generator, không tạo toàn bộ danh sách cặp).

### Cách 3 — Sắp xếp + hai con trỏ

**Từng đoạn:**

- `nums.sort()` (hoặc `a = sorted(nums)` để không phá mảng gốc).
- `l, r = 0, n − 1`; lặp khi `l < r`.
- Tổng `< target`: `count += r − l; l += 1`. Ngược lại: `r -= 1`.

In [14]:
def count_pairs_two_pointers(nums: List[int], target: int) -> int:
    a = sorted(nums)
    l, r = 0, len(a) - 1
    count = 0
    while l < r:
        if a[l] + a[r] < target:
            count += r - l      # a[l] ghép được với mọi phần tử l+1..r
            l += 1
        else:
            r -= 1
    return count

In [15]:
print(count_pairs_two_pointers([-1, 1, 2, 3, 1], 2))              # 3
print(count_pairs_two_pointers([-6, 2, 5, -2, -7, -1, 3], -2))    # 10

3
10


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def countPairs(self, nums: List[int], target: int) -> int:
        nums.sort()
        l, r = 0, len(nums) - 1
        count = 0
        while l < r:
            if nums[l] + nums[r] < target:
                count += r - l
                l += 1
            else:
                r -= 1
        return count
```

**Độ phức tạp:** sắp xếp `O(n log n)`, hai con trỏ `O(n)` ⇒ Time `O(n log n)`, Space `O(1)` (nếu sắp xếp tại chỗ) hoặc `O(n)` (nếu `sorted`).

### Cách 4 — Sắp xếp + `bisect_left`

**Từng đoạn:**

- Sau khi sắp xếp, với mỗi `i`, ta muốn đếm `j > i` với `a[j] < target − a[i]`.
- `bisect_left(a, target - a[i], i + 1)` trả vị trí đầu tiên `j >= i + 1` mà `a[j] >= target − a[i]`. Do đó số `j` hợp lệ là `idx − (i + 1)`.
- Cộng dồn qua mọi `i`.

In [16]:
def count_pairs_bisect(nums: List[int], target: int) -> int:
    a = sorted(nums)
    count = 0
    for i in range(len(a)):
        idx = bisect_left(a, target - a[i], i + 1)   # vị trí đầu tiên có a[j] >= target - a[i]
        count += idx - (i + 1)
    return count

In [17]:
print(count_pairs_bisect([-1, 1, 2, 3, 1], 2))              # 3
print(count_pairs_bisect([-6, 2, 5, -2, -7, -1, 3], -2))    # 10

3
10


**Độ phức tạp:** sắp xếp `O(n log n)`, `n` lần tìm nhị phân mỗi lần `O(log n)` ⇒ Time `O(n log n)`, Space `O(n)` nếu dùng `sorted` (hoặc `O(1)` phụ nếu `sort` tại chỗ).

Cách 4 tương đương cách 3 nhưng dễ nghĩ hơn cho người quen "với mỗi phần tử, đếm các phần tử thoả điều kiện bằng tìm kiếm nhị phân".

## 8. TRACE từng bước

Trace Cách 3 (hai con trỏ) trên `nums = [-6, 2, 5, -2, -7, -1, 3]`, `target = −2`. Sau khi sắp xếp: `a = [-7, -6, -2, -1, 2, 3, 5]`.

| bước | `l` | `r` | `a[l]` | `a[r]` | tổng | so với `−2` | hành động | `count` |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 6 | −7 | 5 | −2 | không `<` | `r -= 1` | 0 |
| 2 | 0 | 5 | −7 | 3 | −4 | `<` | `count += 5 − 0 = 5`, `l += 1` | 5 |
| 3 | 1 | 5 | −6 | 3 | −3 | `<` | `count += 5 − 1 = 4`, `l += 1` | 9 |
| 4 | 2 | 5 | −2 | 3 | 1 | không | `r -= 1` | 9 |
| 5 | 2 | 4 | −2 | 2 | 0 | không | `r -= 1` | 9 |
| 6 | 2 | 3 | −2 | −1 | −3 | `<` | `count += 3 − 2 = 1`, `l += 1` | **10** |
| — | 3 | 3 | | | | `l < r` sai | dừng | 10 |

In [18]:
def trace_two_pointers(nums, target):
    a = sorted(nums)
    print("mảng đã sắp xếp:", a)
    l, r = 0, len(a) - 1
    count = 0
    buoc = 0
    while l < r:
        buoc += 1
        s = a[l] + a[r]
        if s < target:
            count += r - l
            hanh_dong = f"count += {r - l}, l += 1"
            ket_qua = "<"
            nl, nr = l + 1, r
        else:
            hanh_dong = "r -= 1"
            ket_qua = ">="
            nl, nr = l, r - 1
        print(f"bước {buoc}: l={l}, r={r}, {a[l]}+{a[r]}={s} {ket_qua} {target} -> {hanh_dong:22} | count={count}")
        l, r = nl, nr
    print("kết quả:", count)
    return count

trace_two_pointers([-6, 2, 5, -2, -7, -1, 3], -2)

mảng đã sắp xếp: [-7, -6, -2, -1, 2, 3, 5]
bước 1: l=0, r=6, -7+5=-2 >= -2 -> r -= 1                 | count=0
bước 2: l=0, r=5, -7+3=-4 < -2 -> count += 5, l += 1     | count=5
bước 3: l=1, r=5, -6+3=-3 < -2 -> count += 4, l += 1     | count=9
bước 4: l=2, r=5, -2+3=1 >= -2 -> r -= 1                 | count=9
bước 5: l=2, r=4, -2+2=0 >= -2 -> r -= 1                 | count=9
bước 6: l=2, r=3, -2+-1=-3 < -2 -> count += 1, l += 1     | count=10
kết quả: 10


10

In [19]:
trace_two_pointers([-1, 1, 2, 3, 1], 2)

mảng đã sắp xếp: [-1, 1, 1, 2, 3]
bước 1: l=0, r=4, -1+3=2 >= 2 -> r -= 1                 | count=0
bước 2: l=0, r=3, -1+2=1 < 2 -> count += 3, l += 1     | count=3
bước 3: l=1, r=3, 1+2=3 >= 2 -> r -= 1                 | count=3
bước 4: l=1, r=2, 1+1=2 >= 2 -> r -= 1                 | count=3
kết quả: 3


3

In [20]:
# Trace Cách 4 (bisect) cho cùng mảng nhỏ để so sánh
def trace_bisect(nums, target):
    a = sorted(nums)
    print("a =", a)
    count = 0
    for i in range(len(a)):
        need = target - a[i]
        idx = bisect_left(a, need, i + 1)
        add = idx - (i + 1)
        count += add
        print(f"i={i}, a[i]={a[i]:3}, cần a[j] < {need:3}, idx={idx}, cộng {add}, count={count}")
    return count

trace_bisect([-1, 1, 2, 3, 1], 2)

a = [-1, 1, 1, 2, 3]
i=0, a[i]= -1, cần a[j] <   3, idx=4, cộng 3, count=3
i=1, a[i]=  1, cần a[j] <   1, idx=2, cộng 0, count=3
i=2, a[i]=  1, cần a[j] <   1, idx=3, cộng 0, count=3
i=3, a[i]=  2, cần a[j] <   0, idx=4, cộng 0, count=3
i=4, a[i]=  3, cần a[j] <  -1, idx=5, cộng 0, count=3


3

## 9. Code nộp LeetCode

Dùng Cách 3 (sắp xếp + hai con trỏ).

In [21]:
from typing import List

class Solution:
    def countPairs(self, nums: List[int], target: int) -> int:
        nums = sorted(nums)             # không phá mảng của người gọi
        l, r = 0, len(nums) - 1
        count = 0
        while l < r:
            if nums[l] + nums[r] < target:
                count += r - l          # nums[l] ghép được với mọi phần tử l+1..r
                l += 1
            else:
                r -= 1                  # nums[r] quá lớn với mọi l' >= l
        return count

In [22]:
print(Solution().countPairs([-1, 1, 2, 3, 1], 2))              # 3
print(Solution().countPairs([-6, 2, 5, -2, -7, -1, 3], -2))    # 10

3
10


## 10. Vì sao thuật toán đúng

Mảng `a` đã sắp xếp tăng dần. Gọi `P(l, r)` là số cặp `(x, y)` với `l <= x < y <= r` thoả `a[x] + a[y] < target`. Đáp án là `P(0, n−1)`. Hai con trỏ duy trì bất biến: **`count` + `P(l, r)` = đáp án**.

**Trường hợp 1: `a[l] + a[r] < target`.** Với mọi `m` thoả `l < m <= r` ta có `a[m] <= a[r]` nên `a[l] + a[m] <= a[l] + a[r] < target`. Vậy **mọi** cặp có phần tử đầu là `l` trong khoảng `[l, r]` đều thoả: có `r − l` cặp. Chúng bị loại khỏi `P(l, r)` khi ta chuyển sang `P(l+1, r)`, và được cộng vào `count`. Bất biến giữ nguyên: `P(l, r) = (r − l) + P(l+1, r)`.

**Trường hợp 2: `a[l] + a[r] >= target`.** Với mọi `x` thoả `l <= x < r` ta có `a[x] >= a[l]` nên `a[x] + a[r] >= a[l] + a[r] >= target`. Vậy **không** cặp nào chứa `r` (với phần tử đầu trong `[l, r−1]`) thoả. Do đó `P(l, r) = P(l, r−1)`. Bất biến giữ nguyên.

**Dừng:** khi `l >= r` không còn cặp nào, `P = 0`, và `count` chính là đáp án.

**Mỗi cặp được đếm đúng một lần:** cặp `(x, y)` với `x < y` chỉ được đếm khi `l = x` (và `r >= y`) ở trường hợp 1, sau đó `l` tăng nên không đếm lại.

**Sắp xếp giữ nguyên đáp án** như đã nói ở Đột phá 1: ánh xạ giữa vị trí cũ và mới là một song ánh, tập các cặp phần tử `{a, b}` không đổi.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp | `O(n²)` | `O(1)` | oracle |
| 2. `combinations` + `sum` | `O(n²)` | `O(1)` | ngắn, vẫn bậc hai |
| 3. Sort + hai con trỏ | `O(n log n)` | `O(1)` hoặc `O(n)` | **code nộp** |
| 4. Sort + `bisect_left` | `O(n log n)` | `O(1)` hoặc `O(n)` | dễ nghĩ, hơi chậm hơn hai con trỏ |

Phần chi phối là sắp xếp `O(n log n)`; riêng hai con trỏ chỉ `O(n)`. Với `n <= 50` mọi cách đều tức thì.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một phần tử (không có cặp); hai phần tử đúng bằng ngưỡng (`>=` target thì không đếm); tất cả cặp đều thoả; không cặp nào thoả; toàn số âm; có số trùng; `target` rất nhỏ.

In [23]:
impls = {
    "bruteforce": count_pairs_bruteforce,
    "combinations": count_pairs_combinations,
    "two_pointers": count_pairs_two_pointers,
    "bisect": count_pairs_bisect,
    "Solution": Solution().countPairs,
}
cases = [
    ("ví dụ 1", [-1, 1, 2, 3, 1], 2, 3),
    ("ví dụ 2", [-6, 2, 5, -2, -7, -1, 3], -2, 10),
    ("một phần tử", [7], 100, 0),
    ("hai phần tử tổng bằng target", [1, 2], 3, 0),
    ("hai phần tử tổng nhỏ hơn target", [1, 2], 4, 1),
    ("mọi cặp đều thoả", [1, 1, 1, 1], 3, 6),
    ("không cặp nào thoả", [5, 5, 5], 10, 0),
    ("toàn số âm", [-5, -4, -3], -7, 2),
    ("có số trùng", [2, 2, 2, 2], 5, 6),
    ("target nhỏ nhất", [-50, -50, -50], -50, 3),
    ("target rất nhỏ so với mảng", [0, 0, 0], -1, 0),
]
for name, nums, target, exp in cases:
    for k, f in impls.items():
        got = f(list(nums), target)
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 10
PASS một phần tử -> 0
PASS hai phần tử tổng bằng target -> 0
PASS hai phần tử tổng nhỏ hơn target -> 1
PASS mọi cặp đều thoả -> 6
PASS không cặp nào thoả -> 0
PASS toàn số âm -> 2
PASS có số trùng -> 6
PASS target nhỏ nhất -> 3
PASS target rất nhỏ so với mảng -> 0


Kiểm tay vài ca: `[-5, -4, -3]`, `target = −7`: `−5 + −4 = −9 < −7` ✓; `−5 + −3 = −8 < −7` ✓; `−4 + −3 = −7` không `<` ⇒ 2 cặp ✓. `[1, 1, 1, 1]`, `target = 3`: mọi tổng `2 < 3`, có `4·3/2 = 6` cặp ✓.

### 12.2. So sánh các cách trên nhiều input ngẫu nhiên có seed

Input hợp lệ theo constraints: `1 <= n <= 50`, `-50 <= nums[i], target <= 50`.

In [24]:
import random
random.seed(2824)
for _ in range(2000):
    n = random.randint(1, 50)
    nums = [random.randint(-50, 50) for _ in range(n)]
    target = random.randint(-50, 50)
    exp = count_pairs_bruteforce(nums, target)
    for k, f in impls.items():
        assert f(list(nums), target) == exp, (nums, target, k)
print("2000 ca ngẫu nhiên: mọi cách khớp brute force")

2000 ca ngẫu nhiên: mọi cách khớp brute force


### 12.3. Kiểm tính chất

- Đáp án nằm trong `[0, n(n−1)/2]`.
- Đáp án **không giảm** khi `target` tăng.
- Hoán vị `nums` không đổi đáp án.
- Hàm không phá mảng đầu vào (ta dùng `sorted`).

In [25]:
random.seed(28240)
for _ in range(500):
    n = random.randint(1, 50)
    nums = [random.randint(-50, 50) for _ in range(n)]
    target = random.randint(-50, 49)
    copy = nums[:]
    r1 = Solution().countPairs(nums, target)
    assert nums == copy                      # không bị sắp xếp tại chỗ
    assert 0 <= r1 <= n * (n - 1) // 2
    assert Solution().countPairs(nums, target + 1) >= r1
    random.shuffle(copy)
    assert Solution().countPairs(copy, target) == r1
print("tính chất đúng")

tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm cả cặp `(i, i)` hoặc cả `(i, j)` và `(j, i)`

In [26]:
def sai_dem_hai_lan(nums, target):
    count = 0
    for i in range(len(nums)):
        for j in range(len(nums)):          # sai: j bắt đầu từ 0, gồm cả i == j và j < i
            if nums[i] + nums[j] < target:
                count += 1
    return count

print("sai :", sai_dem_hai_lan([-1, 1, 2, 3, 1], 2))              # đếm thừa
print("đúng:", Solution().countPairs([-1, 1, 2, 3, 1], 2))         # 3

sai : 7
đúng: 3


**Vì sao sai:** đề yêu cầu `i < j`. Cho `j` chạy từ `i + 1`. Nếu `j` chạy từ `0`, cặp `(i, j)` và `(j, i)` đếm hai lần, và `(i, i)` còn ghép phần tử với chính nó.

### Lỗi 2: Hai con trỏ nhưng chỉ cộng `1` thay vì `r - l`

In [27]:
def sai_cong_1(nums, target):
    a = sorted(nums)
    l, r = 0, len(a) - 1
    count = 0
    while l < r:
        if a[l] + a[r] < target:
            count += 1               # sai: bỏ sót các cặp (l, m) với l < m < r
            l += 1
        else:
            r -= 1
    return count

print("sai :", sai_cong_1([-6, 2, 5, -2, -7, -1, 3], -2))          # nhỏ hơn 10
print("đúng:", Solution().countPairs([-6, 2, 5, -2, -7, -1, 3], -2))   # 10

sai : 3
đúng: 10


**Vì sao sai:** khi `a[l] + a[r] < target`, **mọi** `m` từ `l+1` đến `r` đều thoả; có `r − l` cặp chứ không phải 1.

### Lỗi 3: Dùng hai con trỏ mà quên sắp xếp

In [28]:
def sai_quen_sort(nums, target):
    l, r = 0, len(nums) - 1
    count = 0
    while l < r:
        if nums[l] + nums[r] < target:
            count += r - l
            l += 1
        else:
            r -= 1
    return count

print("sai :", sai_quen_sort([3, 1, 2, -1, 1], 2))              # mảng chưa sắp xếp
print("đúng:", Solution().countPairs([3, 1, 2, -1, 1], 2))       # 3

sai : 0
đúng: 3


**Vì sao sai:** lập luận "`nums[m] <= nums[r]` với `m <= r`" chỉ đúng khi mảng **tăng dần**. Phải sắp xếp trước.

### Lỗi 4: Dùng `<=` thay vì `<`

In [29]:
def sai_nho_hon_hoac_bang(nums, target):
    return sum(a + b <= target for a, b in combinations(nums, 2))   # sai

print("sai :", sai_nho_hon_hoac_bang([-1, 1, 2, 3, 1], 2))     # đếm cả cặp (0, 3), (1, 4)
print("đúng:", Solution().countPairs([-1, 1, 2, 3, 1], 2))      # 3

sai : 5
đúng: 3


**Vì sao sai:** đề yêu cầu tổng **nhỏ hơn hẳn** `target`. Tổng bằng `target` không được tính.

### Lỗi 5 (nhỏ): `nums.sort()` làm thay đổi mảng đầu vào

Trên LeetCode việc sắp xếp tại chỗ thường chấp nhận được, nhưng trong thực tế (và khi bạn dùng lại mảng sau), nên dùng `sorted(nums)` để không có tác dụng phụ ngoài ý muốn.

In [30]:
a = [3, 1, 2]
b = sorted(a)
print(a, b)        # [3, 1, 2] [1, 2, 3]  : a vẫn nguyên
a.sort()
print(a)           # [1, 2, 3] : a đã bị đổi

[3, 1, 2] [1, 2, 3]
[1, 2, 3]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "đếm số cặp có tổng / hiệu thoả điều kiện kiểu `<`, `>`" và **thứ tự chỉ số không quan trọng** ⇒ **sắp xếp** rồi dùng **hai con trỏ** hoặc **tìm kiếm nhị phân**.
- Mẫu hai con trỏ đếm cặp: khi `a[l] + a[r]` thoả, cộng `r − l` (cả cụm) rồi `l += 1`; không thoả thì `r -= 1`.
- Nếu đề yêu cầu **chỉ số** gốc (như Two Sum) thì không sắp xếp được tuỳ tiện; còn bài đếm cặp thì sắp xếp được.
- `n` nhỏ (`<= 50`) thì brute force đủ; luôn học cách tối ưu để dùng khi `n` lớn.
- Đếm `True` bằng `sum(...)` là mẹo ngắn gọn để đếm số phần tử / cặp thoả điều kiện.

## 15. Biến thể của bài

1. **Tổng lớn hơn target** (`>`): đối xứng, `count += r − l` khi `a[l] + a[r] > target` và `r -= 1`.
2. **Tổng nằm trong đoạn `[lo, hi]`** (xem LeetCode 1099 / 2563): `f(hi) − f(lo − 1)` với `f(t)` = số cặp tổng `<= t`.
3. **Tổng bằng đúng `target`** (Two Sum đếm số cặp): dùng `Counter` hoặc hai con trỏ.
4. **Bộ ba** `i < j < k` có tổng `< target` (3Sum Smaller, LeetCode 259): cố định `i`, hai con trỏ cho phần còn lại.
5. **Hiệu của cặp** `|a − b| < k`: sắp xếp + cửa sổ trượt.
6. **Trả về các cặp** thay vì số lượng: duyệt trực tiếp theo cụm.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp | xét mọi cặp | `O(n²)` | `O(1)` | oracle, `n` nhỏ |
| 2. `combinations` | một dòng, vẫn mọi cặp | `O(n²)` | `O(1)` | code ngắn khi `n` nhỏ |
| 3. Sort + hai con trỏ | đếm cả cụm `r − l` | `O(n log n)` | `O(1)`/`O(n)` | **code nộp**, `n` lớn |
| 4. Sort + `bisect_left` | với mỗi `i`, tìm ranh giới | `O(n log n)` | `O(1)`/`O(n)` | khi quen với tìm nhị phân |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sum(x < 3 for x in [1, 2, 3, 4])`, `sorted([3, -1, 2])`, `bisect_left([1, 3, 5], 4)`, `len(list(combinations(range(6), 2)))`.

**Bài 2 (tính tay).** Với `nums = [2, 4, 1, 3]`, `target = 6`: liệt kê mọi cặp, tổng và đáp án; sau đó chạy hai con trỏ trên mảng đã sắp xếp và đối chiếu.

**Bài 3 (điền TODO).** Hoàn thành `count_pairs_todo(nums, target)` theo hai con trỏ.

In [31]:
def count_pairs_todo(nums, target):
    # TODO 1: a = sorted(nums)
    # TODO 2: l = 0, r = len(a) - 1, count = 0
    # TODO 3: while l < r:
    # TODO 4:     nếu a[l] + a[r] < target: count += r - l, l += 1
    # TODO 5:     ngược lại: r -= 1
    # TODO 6: return count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_pairs_bisect2(nums, target)` bằng `bisect_left` **không nhìn** code mẫu. Thử với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_pairs_greater(nums, target)`: đếm cặp `i < j` có tổng **lớn hơn** `target`. (b) Viết `count_triplets_less(nums, target)` (3Sum Smaller): đếm bộ ba `i < j < k` có tổng `< target`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `x < 3` đúng với `1, 2` nên `2`; `sorted([3, -1, 2]) = [-1, 2, 3]`; `bisect_left([1, 3, 5], 4) = 2`; `6·5/2 = 15`. Bài 2: `[2, 4, 1, 3]` có `6` cặp; chỉ cần đếm những cặp có tổng `< 6`.
- **Mức 2 (vừa):** Bài 3: sáu dòng đã ghi trong comment TODO. Bài 4: với mỗi `i`, `idx = bisect_left(a, target - a[i], i + 1)`, cộng `idx − (i + 1)`.
- **Mức 3 (sâu):** Bài 5(a): đảo điều kiện: nếu `a[l] + a[r] > target` thì mọi `m` từ `l` tới `r−1` ghép được với `r`, cộng `r − l` rồi `r -= 1`; ngược lại `l += 1`. Bài 5(b): sắp xếp, cố định `i`, rồi hai con trỏ `l = i + 1`, `r = n − 1`: nếu `a[i] + a[l] + a[r] < target` thì cộng `r − l`, `l += 1`; ngược lại `r -= 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert sum(x < 3 for x in [1, 2, 3, 4]) == 2
assert sorted([3, -1, 2]) == [-1, 2, 3]
assert bisect_left([1, 3, 5], 4) == 2
assert len(list(combinations(range(6), 2))) == 15

# Bài 2: [2, 4, 1, 3], target = 6
# cặp: (2,4)=6 x, (2,1)=3 v, (2,3)=5 v, (4,1)=5 v, (4,3)=7 x, (1,3)=4 v  -> 4 cặp
assert Solution().countPairs([2, 4, 1, 3], 6) == 4
assert count_pairs_bruteforce([2, 4, 1, 3], 6) == 4

# Bài 3
def count_pairs_todo_ans(nums, target):
    a = sorted(nums)
    l, r = 0, len(a) - 1
    count = 0
    while l < r:
        if a[l] + a[r] < target:
            count += r - l
            l += 1
        else:
            r -= 1
    return count

assert count_pairs_todo_ans([-1, 1, 2, 3, 1], 2) == 3
assert count_pairs_todo_ans([-6, 2, 5, -2, -7, -1, 3], -2) == 10

# Bài 4
def count_pairs_bisect2(nums, target):
    a = sorted(nums)
    total = 0
    for i in range(len(a)):
        idx = bisect_left(a, target - a[i], i + 1)
        total += idx - (i + 1)
    return total

assert count_pairs_bisect2([-1, 1, 2, 3, 1], 2) == 3
assert count_pairs_bisect2([-6, 2, 5, -2, -7, -1, 3], -2) == 10

# Bài 5(a)
def count_pairs_greater(nums, target):
    a = sorted(nums)
    l, r = 0, len(a) - 1
    count = 0
    while l < r:
        if a[l] + a[r] > target:
            count += r - l
            r -= 1
        else:
            l += 1
    return count

# Bài 5(b)
def count_triplets_less(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 2):
        l, r = i + 1, n - 1
        while l < r:
            if a[i] + a[l] + a[r] < target:
                count += r - l
                l += 1
            else:
                r -= 1
    return count

assert count_triplets_less([-2, 0, 1, 3], 2) == 2          # (-2,0,1) và (-2,0,3)

random.seed(7)
for _ in range(400):
    nums = [random.randint(-50, 50) for _ in range(random.randint(1, 30))]
    t = random.randint(-50, 50)
    assert count_pairs_todo_ans(nums, t) == count_pairs_bisect2(nums, t) == count_pairs_bruteforce(nums, t)
    exp_g = sum(a + b > t for a, b in combinations(nums, 2))
    assert count_pairs_greater(nums, t) == exp_g
    exp_3 = sum(a + b + c < t for a, b, c in combinations(nums, 3))
    assert count_triplets_less(nums, t) == exp_3
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cặp tổng `< target`: sắp xếp + hai con trỏ)

```python
from typing import List

class Solution:
    def countPairs(self, nums: List[int], target: int) -> int:
        nums = sorted(nums)
        l, r = 0, len(nums) - 1
        count = 0
        while l < r:
            if nums[l] + nums[r] < target:
                count += r - l        # nums[l] ghép được với mọi phần tử l+1..r
                l += 1
            else:
                r -= 1                # nums[r] quá lớn
        return count
```

**Checklist**

- [ ] Cặp `i < j` (không đếm `(i, i)`, không đếm hai lần)?
- [ ] Điều kiện là `<` (không phải `<=`)?
- [ ] Đã sắp xếp trước khi dùng hai con trỏ?
- [ ] Khi thoả điều kiện: cộng `r − l` (không phải `1`)?
- [ ] Đã thử: `n = 1`, tất cả cặp thoả, không cặp nào thoả, số âm, số trùng?
- [ ] Đã so với brute force trên input ngẫu nhiên có seed?
- [ ] Signature đúng `countPairs(self, nums: List[int], target: int) -> int`?